In [1]:
import pandas as pd
import numpy as np
import os

# create processed folder if it dosen't exist
os.makedirs('../data/processed', exist_ok=True)

### 1. Clean investor_transactions.csv

In [2]:
# load raw data
df_nav = pd.read_csv('../data/raw/02_nav_history.csv')
print(df_nav)

# 1. Parse dates to datetime
df_nav['date'] = pd.to_datetime(df_nav['date'])

# 2. Sort by amfi_code and date
df_nav = df_nav.sort_values(by=['amfi_code', 'date'])

# 3. Remove duplicates
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'])

# 4. Validate NAV > 0
df_nav = df_nav[df_nav['nav']>0]

# 5. Forward-fill missing NAV for holidays/weekends (Crucial Step!)
def reindex_and_ffill(group):
    # FIX: Get the amfi_code from the group's 'name' attribute instead of the column
    current_amfi_code = group.name 
    
    group = group.set_index('date')
    
    # Create a complete date range from min date to max date for this specific fund
    full_date_range = pd.date_range(start=group.index.min(), end=group.index.max(), freq='D' )
    group = group.reindex(full_date_range)
    
    # Re-assign the amfi_code and forward-fill the nav
    group['amfi_code'] = current_amfi_code
    group['nav'] = group['nav'].ffill()
    
    return group.rename_axis('date').reset_index()

# Apply the function (Warning is silenced, and no KeyError!)
df_nav_clean = df_nav.groupby('amfi_code').apply(reindex_and_ffill, include_groups=False).reset_index(drop=True)

# Save to processed folder
df_nav_clean.to_csv('../data/processed/clean_nav_history.csv', index=False)
print("nav_history.csv cleaned and saved!")
    

       amfi_code        date       nav
0         119551  2022-01-03   54.3856
1         119551  2022-01-04   54.3474
2         119551  2022-01-05   54.6869
3         119551  2022-01-06   55.4550
4         119551  2022-01-07   55.3692
...          ...         ...       ...
45995     149324  2026-05-25  292.4810
45996     149324  2026-05-26  291.2707
45997     149324  2026-05-27  288.8007
45998     149324  2026-05-28  280.6873
45999     149324  2026-05-29  279.7511

[46000 rows x 3 columns]
nav_history.csv cleaned and saved!


### 2. Clean investor_transactions.csv

In [3]:
df_tx = pd.read_csv('../data/raw/08_investor_transactions.csv')

# 1. Fix date formats
df_tx['transaction_date'] = pd.to_datetime(df_tx['transaction_date'])

# 2. Standardise transaction_type values (e.g., 'sip', ' SIP ', 'Sip' -> 'SIP')
df_tx['transaction_type'] = df_tx['transaction_type'].str.strip().str.title()
# Make sure 'Sip' becomes 'SIP' for consistency
df_tx['transaction_type'] = df_tx['transaction_type'].replace('Sip','SIP')

# 3. Validate amount > 0
df_tx = df_tx[df_tx['amount_inr']>0]

# 4. Check KYC status enum values (Only keep Verified or Pending)
df_tx['kyc_status'] = df_tx['kyc_status'].str.strip().str.title()
valid_kyc = ['Verified', 'Pending']
df_tx = df_tx[df_tx['kyc_status'].isin(valid_kyc)]

# Save to processed folder
df_tx.to_csv('../data/processed/clean_transaction.csv', index=False)
print("investor_transactions.csv cleaned and saved")


investor_transactions.csv cleaned and saved


### 3. Clean scheme_performance.csv

In [4]:
df_perf =pd.read_csv('../data/raw/07_scheme_performance.csv')

# 1. Validate all return values are numeric (converts errors to NaN)
return_cols = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'sharpe_ratio', 'expense_ratio_pct']
for col in return_cols:
    if col in df_perf.columns:
        df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

# 2. Flag anomalies (e.g., negative Sharpe ratio)
# Creates a new column 'is_anomaly' which is True if Sharpe < 0
if 'sharpe_ratio_pct' in df_perf.columns:
    df_perf['is_anomaly'] = np.where(df_perf['sharpe_ratio'] < 0, True, False)
    
# 3. Check expense_ratio range (0.1% – 2.5%)
# Keep only rows where expense ratio is valid
if 'expense_ratio_pct' in df_perf.columns:
    df_perf = df_perf[df_perf['expense_ratio_pct'].between(0.1, 2.5)]
    
# save to processed folder
df_perf.to_csv('../data/processed/clean_performance.csv', index=False)
print("scheme_performance.csv cleaned and saved")

scheme_performance.csv cleaned and saved


In [5]:
import sqlite3
import os

# Create db folder if it doesn't exist
os.makedirs('../data/db', exist_ok=True)

# Create db folder if it doesn't exist
db_path ='../data/db/bluestock_mf.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Read the schema.sql file and execute it
with open('../sql/schema.sql', 'r', encoding='utf-8') as file:
    sql_script = file.read()

# Execute the SQL script to create tables
cursor.executescript(sql_script)

# Commit and close
conn.commit()
conn.close()

print("Database 'bluestock_mf.db' and all tables created successfully")


Database 'bluestock_mf.db' and all tables created successfully


In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# 1. Create SQLAlchemy engine
engine = create_engine('sqlite:///../data/db/bluestock_mf.db')
print("Loading data into the database... Please wait.")

# 2. Load CSV files as Pandas DataFrames
df_fund = pd.read_csv('../data/raw/01_fund_master.csv')
df_nav = pd.read_csv('../data/processed/clean_nav_history.csv')
df_tx = pd.read_csv('../data/processed/clean_transaction.csv')
df_perf = pd.read_csv('../data/processed/clean_performance.csv')
df_aum = pd.read_csv('../data/raw/03_aum_by_fund_house.csv')

# 3. Load DataFrames into database tables

# Load dim_fund
fund_columns_to_keep = ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'risk_category']
df_fund_subset = df_fund[fund_columns_to_keep]
df_fund_subset.to_sql('dim_fund', con=engine, if_exists='append', index=False)
print(f"Loaded {len(df_fund_subset)} rows into dim_fund")

# Load fact_nav
df_nav = df_nav.rename(columns={'date': 'nav_date'})
df_nav.to_sql('fact_nav', con=engine, if_exists='append', index=False)
print(f"Loaded {len(df_nav)} rows into fact_nav")

# Load fact_transactions
tx_columns_to_keep = ['investor_id', 'amfi_code', 'transaction_date', 'transaction_type', 'amount_inr', 'kyc_status']
df_tx_subset = df_tx[tx_columns_to_keep]
df_tx_subset.to_sql('fact_transactions', con=engine, if_exists='append', index=False)
print(f"Loaded {len(df_tx_subset)} rows into fact_transactions")

# Load fact_performance
perf_columns_to_keep = ['amfi_code', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'sharpe_ratio', 'expense_ratio_pct', 'is_anomaly']
if 'is_anomaly' not in df_perf.columns:
    df_perf['is_anomaly'] = False 
df_perf_subset = df_perf[perf_columns_to_keep]
df_perf_subset.to_sql('fact_performance', con=engine, if_exists='append', index=False)
print(f"Loaded {len(df_perf_subset)} rows into fact_performance")

# Load fact_aum
df_aum = df_aum.rename(columns={'quarter': 'quarter_date', 'date': 'quarter_date', 'aum': 'aum_crore'})
aum_columns_to_keep = ['fund_house', 'quarter_date', 'aum_crore']
df_aum_subset = df_aum[aum_columns_to_keep]
df_aum_subset.to_sql('fact_aum', con=engine, if_exists='append', index=False)
print(f"Loaded {len(df_aum_subset)} rows into fact_aum")

print("All datasets successfully loaded into SQLite Database!")

Loading data into the database... Please wait.
Loaded 40 rows into dim_fund
Loaded 64320 rows into fact_nav
Loaded 32778 rows into fact_transactions
Loaded 40 rows into fact_performance
Loaded 90 rows into fact_aum
🎉 All datasets successfully loaded into SQLite Database!
